# ⚡ Module 14: LoRA (Low-Rank Adaptation) from Scratch
## 🎓 Fine-Tuning 70B Models with 10,000x Fewer Parameters (Hu et al. 2021)
*Author: Muhammad Zabiullah ([@mzb000](https://github.com/mzb000))*

---

## 📌 Table of Contents
1. **The Fine-Tuning Wall: Storing Full Gradients for Billion-Parameter Weights**
2. **The Intrinsic Rank Hypothesis: Task Changes Lie in Low-Dimensional Subspaces**
3. **The LoRA Decomposition:**
   $$W_{\text{new}} = W_0 + \Delta W = W_0 + \frac{\alpha}{r} (B \cdot A)$$
   Where $W_0$ is frozen $(d \times k)$, $A$ is $(r \times k)$, and $B$ is $(d \times r)$ with rank $r \ll d$.
4. **Hands-On Parameter Comparison in Pure NumPy**

In [ ]:
import numpy as np

# Layer Dimension: d = 4096 (Standard LLaMA layer size)
d_in, d_out = 4096, 4096
rank = 8  # Low rank bottleneck
alpha = 16

# Standard Full Weight Matrix (Frozen during LoRA!)
full_params = d_in * d_out

# LoRA Adapters: A (r x d_in), B (d_out x r)
lora_params = (rank * d_in) + (d_out * rank)
reduction = (1.0 - (lora_params / full_params)) * 100

print(f"Standard Layer Parameters : {full_params:,} weights (All require optimizer states & gradients)")
print(f"LoRA Adapter Parameters    : {lora_params:,} weights (Rank r={rank})")
print(f"Trainable Parameter Cut    : {reduction:.2f}% REDUCTION IN TRAINABLE WEIGHTS! 🚀")

# Forward Pass Demo
W_frozen = np.random.randn(d_in, d_out) * 0.02
A_adapter = np.random.randn(d_in, rank) * 0.01
B_adapter = np.zeros((rank, d_out))  # Initialized to ZERO so training starts with exact base model!

x_input = np.random.randn(1, d_in)
# Output = x @ W_0 + (alpha/r) * (x @ A @ B)
base_output = x_input @ W_frozen
lora_output = base_output + (alpha / rank) * ((x_input @ A_adapter) @ B_adapter)

print("\nVerification: Output with zero-initialized B equals base output exactly!")
print("Max absolute difference:", np.max(np.abs(base_output - lora_output)))